<a href="https://colab.research.google.com/github/CodeWithradhika/CSET343./blob/main/Lab7_DL_GridSearch_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSET343 - Lab Assignment 7
## Deep Learning Grid Search (Keras + Scikit-learn) on Pima Indians Diabetes

Note: scikeras is not used because it breaks with the new scikit-learn on Colab (`__sklearn_tags__` error). A small custom sklearn-compatible Keras wrapper is used instead.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)

## Task 1: Data Acquisition and Loading

In [ ]:
COLUMNS = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
           "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
df = pd.read_csv(URL, header=None, names=COLUMNS)

print("Shape:", df.shape)
display(df.head())
print("\nClass distribution:\n", df["Outcome"].value_counts())

## Task 2: Data Preprocessing

In [ ]:
# 0 is impossible in these columns -> treat as missing
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)
print("Missing values:\n", df.isnull().sum())

# Fill with median
df[zero_as_missing] = df[zero_as_missing].fillna(df[zero_as_missing].median())

X = df.drop("Outcome", axis=1).values
y = df["Outcome"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

# Scale (fit on train only -> no data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("\nTrain:", X_train.shape, "| Test:", X_test.shape)

## Task 3: Deep Learning Grid Search
### Keras model + scikit-learn wrapper

In [ ]:
def create_model(input_dim, hidden_units, activation, init_mode,
                 dropout_rate, learning_rate):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(hidden_units, activation=activation, kernel_initializer=init_mode),
        layers.Dropout(dropout_rate),
        layers.Dense(8, activation=activation, kernel_initializer=init_mode),
        layers.Dense(1, activation="sigmoid", kernel_initializer=init_mode),
    ])
    model.compile(loss="binary_crossentropy",
                  optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=["accuracy"])
    return model


class KerasClassifierWrapper(ClassifierMixin, BaseEstimator):
    """Minimal scikit-learn compatible wrapper around a Keras model."""
    def __init__(self, hidden_units=12, activation="relu",
                 init_mode="glorot_uniform", dropout_rate=0.0,
                 learning_rate=0.001, epochs=50, batch_size=16):
        self.hidden_units = hidden_units
        self.activation = activation
        self.init_mode = init_mode
        self.dropout_rate = dropout_rate
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

    def fit(self, X, y):
        X = np.asarray(X, dtype="float32")
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.model_ = create_model(X.shape[1], self.hidden_units, self.activation,
                                   self.init_mode, self.dropout_rate,
                                   self.learning_rate)
        self.model_.fit(X, y, epochs=self.epochs,
                        batch_size=self.batch_size, verbose=0)
        return self

    def predict(self, X):
        probs = self.model_.predict(np.asarray(X, dtype="float32"), verbose=0)
        return (probs > 0.5).astype(int).ravel()

### Helper: run one grid search and carry the best params forward

In [ ]:
best_params = dict(hidden_units=12, activation="relu", init_mode="glorot_uniform",
                   dropout_rate=0.0, learning_rate=0.001, epochs=50, batch_size=16)

def run_grid_search(title, param_grid):
    print("\n" + "=" * 60)
    print("Tuning:", title)
    print("=" * 60)

    clf = KerasClassifierWrapper(**best_params)
    grid = GridSearchCV(clf, param_grid, cv=3, scoring="accuracy", n_jobs=1)
    result = grid.fit(X_train, y_train)

    print(f"Best: {result.best_score_:.4f} using {result.best_params_}")
    for m, s, p in zip(result.cv_results_["mean_test_score"],
                       result.cv_results_["std_test_score"],
                       result.cv_results_["params"]):
        print(f"  {m:.4f} ({s:.4f}) with: {p}")

    best_params.update(result.best_params_)   # carry forward
    return result

### 1) Batch size and training epochs

In [ ]:
r1 = run_grid_search("Batch Size & Epochs",
    {"batch_size": [10, 20, 40, 60, 80, 100],
     "epochs": [10, 50, 100]})

### 2) Learning rate

In [ ]:
r2 = run_grid_search("Learning Rate",
    {"learning_rate": [0.0001, 0.001, 0.01, 0.1]})

### 3) Network weight initialization

In [ ]:
r3 = run_grid_search("Weight Initialization",
    {"init_mode": ["random_uniform", "lecun_uniform", "random_normal", "zeros",
                   "glorot_normal", "glorot_uniform", "he_normal", "he_uniform"]})

### 4) Activation function

In [ ]:
r4 = run_grid_search("Activation Function",
    {"activation": ["softmax", "softplus", "softsign", "relu",
                    "tanh", "sigmoid", "hard_sigmoid", "linear"]})

### 5) Dropout regularization

In [ ]:
r5 = run_grid_search("Dropout Rate",
    {"dropout_rate": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]})

### 6) Number of neurons in the hidden layer

In [ ]:
r6 = run_grid_search("Neurons in Hidden Layer",
    {"hidden_units": [1, 5, 10, 15, 20, 25, 30]})

## Final model with all best hyperparameters

In [ ]:
print("FINAL BEST HYPERPARAMETERS")
for k, v in best_params.items():
    print(f"  {k}: {v}")

final_clf = KerasClassifierWrapper(**best_params)
final_clf.fit(X_train, y_train)

y_pred = final_clf.predict(X_test)
print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))